In [ ]:
import numpy as np

def wrap(x):
    return (x + 2*np.pi) % (2*np.pi)

def psi_singlet_like(xA, xB):
    # Psi ~ sin(xA - xB)
    return np.sin(xA - xB) + 1e-9  # avoid zero

def phase_S(xA, xB):
    # For real Psi this is tricky (phase jumps). Use a complex Psi instead:
    # Example complex entangled wave:
    Psi = np.exp(1j*(xA-xB)) - np.exp(-1j*(xA-xB))  # 2j sin(xA-xB)
    return np.angle(Psi)

def grad_S(xA, xB, eps=1e-4):
    # finite difference on torus
    S0 = phase_S(xA, xB)
    SA = phase_S(wrap(xA+eps), xB)
    SB = phase_S(xA, wrap(xB+eps))
    dSdxA = np.angle(np.exp(1j*(SA - S0))) / eps
    dSdxB = np.angle(np.exp(1j*(SB - S0))) / eps
    return dSdxA, dSdxB

def g_local(x, setting):
    # local "analyzer coupling": choose something sinusoidal in relative angle
    return np.cos(x - setting)

def step(xA, xB, yA, yB, a, b, dt=1e-3, omega=1.0, kappa=0.2, eta=2.0):
    dSdxA, dSdxB = grad_S(xA, xB)

    # rotor guidance (nonlocal via grad S)
    xA = wrap(xA + dt*(omega + kappa*dSdxA))
    xB = wrap(xB + dt*(omega + kappa*dSdxB))

    # bistable pointer dynamics
    # double well potential derivative: d/dy [(y^2-1)^2/4] = y*(y^2-1)
    yA = yA + dt*(-yA*(yA**2 - 1) + eta*g_local(xA, a))
    yB = yB + dt*(-yB*(yB**2 - 1) + eta*g_local(xB, b))

    return xA, xB, yA, yB

def run_trial(a, b, T=2.0, dt=1e-3, rng=None):
    rng = rng or np.random.default_rng()
    # initialize x from |Psi|^2 equilibrium (rough rejection sampling)
    # (for speed: just uniform init for now; later sample from |Psi|^2)
    xA = rng.uniform(0, 2*np.pi)
    xB = rng.uniform(0, 2*np.pi)
    yA = rng.normal(scale=0.1)
    yB = rng.normal(scale=0.1)

    n = int(T/dt)
    for _ in range(n):
        xA, xB, yA, yB = step(xA, xB, yA, yB, a, b, dt=dt)

    sA = +1 if yA >= 0 else -1
    sB = +1 if yB >= 0 else -1
    return sA, sB

def estimate_E(a, b, N=2000, seed=0):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(N):
        sA, sB = run_trial(a, b, rng=rng)
        vals.append(sA*sB)
    return np.mean(vals)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Pauli + analyzers
# ----------------------------
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)

def observable(angle):
    # polarization-like double-angle mapping:
    # O(a) = cos(2a) σz + sin(2a) σx
    return np.cos(2*angle) * sz + np.sin(2*angle) * sx

def eig_projectors(O):
    w, V = np.linalg.eigh(O)
    idx_plus = np.argmax(w)
    idx_minus = 1 - idx_plus
    v_plus = V[:, idx_plus]
    v_minus = V[:, idx_minus]
    P_plus = np.outer(v_plus, v_plus.conj())
    P_minus = np.outer(v_minus, v_minus.conj())
    return (+1, v_plus, P_plus), (-1, v_minus, P_minus)

def bell_singlet():
    # |psi-> = (|01> - |10>)/sqrt(2) in basis |00>,|01>,|10>,|11>
    psi = np.zeros(4, dtype=complex)
    psi[1] = 1/np.sqrt(2)
    psi[2] = -1/np.sqrt(2)
    return psi

def joint_table(psi, a, b):
    (sAp, vAp, PAp), (sAm, vAm, PAm) = eig_projectors(observable(a))
    (sBp, vBp, PBp), (sBm, vBm, PBm) = eig_projectors(observable(b))

    outcomes = []
    probs = []
    for (sA, vA, PA) in [(sAp, vAp, PAp), (sAm, vAm, PAm)]:
        for (sB, vB, PB) in [(sBp, vBp, PBp), (sBm, vBm, PBm)]:
            P = np.kron(PA, PB)
            p = np.vdot(psi, P @ psi).real
            outcomes.append((sA, sB, vA, vB, P))
            probs.append(p)

    probs = np.array(probs, dtype=float)
    probs = probs / probs.sum()
    return outcomes, probs

# ----------------------------
# Bohmian-style deterministic sampling
# ----------------------------
def pilot_sample_joint_outcome(psi, a, b, u):
    """
    Bohmian-inspired "deterministic" outcome rule:

    - u in [0,1) is the hidden variable (initial configuration / pointer position).
    - The pilot wave provides Born weights p_k(a,b) via |Psi|^2.
    - Outcome is deterministic given (u, a, b): pick k by inverse CDF.

    This is explicitly NONLOCAL because p_k depends on BOTH a and b.
    """
    outcomes, probs = joint_table(psi, a, b)
    cdf = np.cumsum(probs)
    k = int(np.searchsorted(cdf, u, side="right"))
    if k >= len(outcomes):
        k = len(outcomes) - 1

    sA, sB, vA, vB, P = outcomes[k]

    # Post-measurement (for visualization only)
    psi_post = P @ psi
    n = np.linalg.norm(psi_post)
    if n > 0:
        psi_post = psi_post / n

    return dict(
        sA=sA, sB=sB,
        vA=vA, vB=vB,
        probs=probs,
        psi_post=psi_post
    )

# ----------------------------
# 3-phase rendering layer
# ----------------------------
def clarke_inv(v_alpha, v_beta):
    va = v_alpha
    vb = -0.5*v_alpha + (np.sqrt(3)/2)*v_beta
    vc = -0.5*v_alpha - (np.sqrt(3)/2)*v_beta
    return va, vb, vc

def qubit_to_alphabeta(q, t, omega=1.0):
    # embed ket q=[q_plus,q_minus] into counter-rotating space vectors
    q_plus, q_minus = q[0], q[1]
    v = q_plus*np.exp(1j*omega*t) + q_minus*np.exp(-1j*omega*t)
    return v.real, v.imag

def qubit_to_3phase(q, t, omega=1.0):
    v_alpha, v_beta = qubit_to_alphabeta(q, t, omega=omega)
    va, vb, vc = clarke_inv(v_alpha, v_beta)
    return (va, vb, vc), (v_alpha, v_beta)

# ----------------------------
# CHSH test
# ----------------------------
def estimate_E(psi, a, b, N=20000, seed=0):
    rng = np.random.default_rng(seed)
    vals = np.empty(N, dtype=float)
    for k in range(N):
        u = rng.random()
        r = pilot_sample_joint_outcome(psi, a, b, u)
        vals[k] = r["sA"] * r["sB"]
    return vals.mean()

def run_chsh(N=40000, seed=1):
    psi = bell_singlet()
    a0, a1 = 0.0, np.pi/4
    b0, b1 = np.pi/8, -np.pi/8

    E00 = estimate_E(psi, a0, b0, N=N, seed=seed+0)
    E01 = estimate_E(psi, a0, b1, N=N, seed=seed+1)
    E10 = estimate_E(psi, a1, b0, N=N, seed=seed+2)
    E11 = estimate_E(psi, a1, b1, N=N, seed=seed+3)

    S = abs(E00 + E01 + E10 - E11)
    return S, (E00, E01, E10, E11)

S, (E00, E01, E10, E11) = run_chsh(N=60000, seed=10)
print("Bohmian-style pilot sampler (deterministic given u)")
print(f"E(a0,b0)={E00:+.4f}  E(a0,b1)={E01:+.4f}  E(a1,b0)={E10:+.4f}  E(a1,b1)={E11:+.4f}")
print(f"CHSH S = {S:.4f}   (2√2 = {2*np.sqrt(2):.4f})")

# ----------------------------
# Visualize one run: probs + resulting abc waveforms
# ----------------------------
psi = bell_singlet()
a = 0.0
b = np.pi/8
u = 0.37  # choose any hidden variable

res = pilot_sample_joint_outcome(psi, a, b, u)
sA, sB = res["sA"], res["sB"]

# render post-outcome local eigenkets as 3-phase
N = 1200
t = np.linspace(0, 2*np.pi, N, endpoint=False)
omega = 1.0

(alice_abc, alice_ab) = qubit_to_3phase(res["vA"], t, omega=omega)
(bob_abc, bob_ab)     = qubit_to_3phase(res["vB"], t, omega=omega)

fig, axs = plt.subplots(2, 2, figsize=(14, 8))

# joint probs bar
labels = ["(+,+)", "(+,-)", "(-,+)", "(-,-)"]
axs[0,0].bar(range(4), res["probs"])
axs[0,0].set_xticks(range(4))
axs[0,0].set_xticklabels(labels)
axs[0,0].set_title(f"Born joint probs p(sA,sB | a,b)\n(a={np.degrees(a):.1f}°, b={np.degrees(b):.1f}°), u={u:.2f} -> (sA,sB)=({sA:+d},{sB:+d})")
axs[0,0].grid(True, alpha=0.3)

# αβ plots
vA_alpha, vA_beta = alice_ab
vB_alpha, vB_beta = bob_ab
axs[0,1].plot(vA_alpha, vA_beta, lw=1.0)
axs[0,1].set_aspect('equal', adjustable='box')
axs[0,1].grid(True, alpha=0.3)
axs[0,1].set_title(f"Alice αβ after outcome (sA={sA:+d})")
axs[0,1].set_xlabel("vα"); axs[0,1].set_ylabel("vβ")

axs[1,1].plot(vB_alpha, vB_beta, lw=1.0)
axs[1,1].set_aspect('equal', adjustable='box')
axs[1,1].grid(True, alpha=0.3)
axs[1,1].set_title(f"Bob αβ after outcome (sB={sB:+d})")
axs[1,1].set_xlabel("vα"); axs[1,1].set_ylabel("vβ")

# abc waveforms (Alice only to keep it readable)
vaA, vbA, vcA = alice_abc
axs[1,0].plot(t, vaA, lw=1.0, label="Va")
axs[1,0].plot(t, vbA, lw=1.0, label="Vb")
axs[1,0].plot(t, vcA, lw=1.0, label="Vc")
axs[1,0].grid(True, alpha=0.3)
axs[1,0].legend(frameon=False, fontsize=8)
axs[1,0].set_title("Alice abc after outcome (render of eigenket)")
axs[1,0].set_xlabel("t")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1) Qubit/Pauli machinery
# ==========================================
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)

def observable(angle):
    # polarization-style: double-angle map
    return np.cos(2*angle) * sz + np.sin(2*angle) * sx

def eigvecs(angle):
    O = observable(angle)
    w, V = np.linalg.eigh(O)
    idx_plus = np.argmax(w)
    idx_minus = 1 - idx_plus
    v_plus = V[:, idx_plus]
    v_minus = V[:, idx_minus]

    # fix global phase for determinism
    if v_plus[0] != 0:
        v_plus *= np.exp(-1j*np.angle(v_plus[0]))
    if v_minus[0] != 0:
        v_minus *= np.exp(-1j*np.angle(v_minus[0]))
    return {+1: v_plus, -1: v_minus}

def bell_singlet():
    # |psi-> = (|01> - |10>)/sqrt(2) in |00>,|01>,|10>,|11>
    psi = np.zeros(4, dtype=complex)
    psi[1] = 1/np.sqrt(2)
    psi[2] = -1/np.sqrt(2)
    return psi

def branch_coeffs_and_kets(psi, a, b):
    """
    Express |psi> in the measurement eigenbases at (a,b):
      c_{sA,sB} = <sA_a, sB_b | psi>
    Returns branches in order: (+,+), (+,-), (-,+), (-,-)
    """
    Psi = psi.reshape(2,2)  # Psi[iA,iB]
    EA = eigvecs(a)
    EB = eigvecs(b)

    branches = []
    coeffs = []
    for sA in [+1, -1]:
        for sB in [+1, -1]:
            vA = EA[sA]
            vB = EB[sB]
            amp = np.sum(np.conj(vA)[:,None] * np.conj(vB)[None,:] * Psi)
            branches.append((sA, sB, vA, vB))
            coeffs.append(amp)
    coeffs = np.array(coeffs, dtype=complex)
    probs = np.abs(coeffs)**2
    probs = probs / probs.sum()
    return branches, coeffs, probs

# ==========================================
# 2) Pointer wavefunction + Bohmian velocity
# ==========================================
def gaussian_packet(y, mu, sigma, k=0.0):
    return np.exp(-(y-mu)**2/(4*sigma**2)) * np.exp(1j*k*y)

def psi_pointer(yA, yB, coeffs, musA, musB, sigma, ksA, ksB):
    Psi = 0.0 + 0.0j
    for c, mA, mB, kA, kB in zip(coeffs, musA, musB, ksA, ksB):
        Psi += c * gaussian_packet(yA, mA, sigma, kA) * gaussian_packet(yB, mB, sigma, kB)
    return Psi

def dpsi_dy(y, mu, sigma, k):
    # derivative of packet wrt y: packet * [-(y-mu)/(2σ^2) + i k]
    p = gaussian_packet(y, mu, sigma, k)
    return p * (-(y-mu)/(2*sigma**2) + 1j*k)

def dpsi_dyA(yA, yB, coeffs, musA, musB, sigma, ksA, ksB):
    dPsi = 0.0 + 0.0j
    for c, mA, mB, kA, kB in zip(coeffs, musA, musB, ksA, ksB):
        dPsi += c * dpsi_dy(yA, mA, sigma, kA) * gaussian_packet(yB, mB, sigma, kB)
    return dPsi

def dpsi_dyB(yA, yB, coeffs, musA, musB, sigma, ksA, ksB):
    dPsi = 0.0 + 0.0j
    for c, mA, mB, kA, kB in zip(coeffs, musA, musB, ksA, ksB):
        dPsi += c * gaussian_packet(yA, mA, sigma, kA) * dpsi_dy(yB, mB, sigma, kB)
    return dPsi

def bohm_velocity(yA, yB, coeffs, musA, musB, sigma, ksA, ksB, eps=1e-12):
    Psi = psi_pointer(yA, yB, coeffs, musA, musB, sigma, ksA, ksB)
    if abs(Psi) < eps:
        return 0.0, 0.0
    vA = np.imag(dpsi_dyA(yA, yB, coeffs, musA, musB, sigma, ksA, ksB) / Psi)
    vB = np.imag(dpsi_dyB(yA, yB, coeffs, musA, musB, sigma, ksA, ksB) / Psi)
    return vA, vB

def pointer_centers(t, branch_signs, sep=3.0, tau=1.2):
    # separation ramp
    mu = sep * (1 - np.exp(-t/tau))
    musA = np.array([sA*mu for (sA,sB) in branch_signs], dtype=float)
    musB = np.array([sB*mu for (sA,sB) in branch_signs], dtype=float)
    return musA, musB

def pointer_phase_slopes(branch_signs, kphase=1.0):
    # simple “push” direction per branch
    ksA = np.array([sA*kphase for (sA,sB) in branch_signs], dtype=float)
    ksB = np.array([sB*kphase for (sA,sB) in branch_signs], dtype=float)
    return ksA, ksB

# ==========================================
# 3) Sampling initial microstates from |Psi|^2
# ==========================================
def sample_initial_from_density(coeffs, branch_signs, sigma, rng, n_try=20000):
    """
    Approx sample (yA0,yB0) from |Psi(yA,yB,t~0)|^2.
    At t=0, all centers are ~0; distribution is basically a mixture.
    We'll do rejection on a grid-free proposal (Gaussian).
    """
    # Proposal: independent N(0, (2*sigma)^2)
    prop_sigma = 2.0*sigma

    # Precompute "envelope" upper bound by scanning a few points
    # (cheap but works ok for this toy)
    musA0 = np.zeros(4)
    musB0 = np.zeros(4)
    ksA0, ksB0 = pointer_phase_slopes(branch_signs, kphase=0.0)

    # quick bound estimate
    ys = rng.normal(0, prop_sigma, size=(2000,2))
    vals = []
    for yA,yB in ys:
        Psi = psi_pointer(yA,yB,coeffs,musA0,musB0,sigma,ksA0,ksB0)
        vals.append((abs(Psi)**2))
    M = 1.2*max(vals) + 1e-12

    for _ in range(n_try):
        yA = rng.normal(0, prop_sigma)
        yB = rng.normal(0, prop_sigma)
        Psi = psi_pointer(yA,yB,coeffs,musA0,musB0,sigma,ksA0,ksB0)
        p = abs(Psi)**2
        if rng.random() < p / M:
            return yA, yB

    # fallback
    return rng.normal(0, prop_sigma), rng.normal(0, prop_sigma)

# ==========================================
# 4) One trial (deterministic given initial microstate)
# ==========================================
def run_trial(psi, a, b, *, T=6.0, dt=0.003, sigma=0.7, sep=3.2, kphase=1.1, seed=None):
    rng = np.random.default_rng(seed)
    branches, coeffs, probs = branch_coeffs_and_kets(psi, a, b)
    branch_signs = [(sA,sB) for (sA,sB,_,_) in branches]

    # sample initial pointer position from |Psi|^2 (“quantum equilibrium”)
    yA, yB = sample_initial_from_density(coeffs, branch_signs, sigma, rng)

    n = int(T/dt)
    ts = np.linspace(0, T, n+1)
    yA_tr = np.empty(n+1); yB_tr = np.empty(n+1)
    yA_tr[0] = yA; yB_tr[0] = yB

    for k, t in enumerate(ts[1:], start=1):
        musA, musB = pointer_centers(t, branch_signs, sep=sep)
        ksA, ksB = pointer_phase_slopes(branch_signs, kphase=kphase)
        vA, vB = bohm_velocity(yA, yB, coeffs, musA, musB, sigma, ksA, ksB)
        yA += dt*vA
        yB += dt*vB
        yA_tr[k] = yA
        yB_tr[k] = yB

    sA = +1 if yA_tr[-1] >= 0 else -1
    sB = +1 if yB_tr[-1] >= 0 else -1

    return dict(
        sA=sA, sB=sB,
        probs=probs,
        branches=branches,
        yA0=yA_tr[0], yB0=yB_tr[0],
        ts=ts, yA_tr=yA_tr, yB_tr=yB_tr
    )

# ==========================================
# 5) 3-phase rendering layer (visual only)
# ==========================================
def clarke_inv(v_alpha, v_beta):
    va = v_alpha
    vb = -0.5*v_alpha + (np.sqrt(3)/2)*v_beta
    vc = -0.5*v_alpha - (np.sqrt(3)/2)*v_beta
    return va, vb, vc

def qubit_to_alphabeta(q, t, omega=1.0):
    q_plus, q_minus = q[0], q[1]
    v = q_plus*np.exp(1j*omega*t) + q_minus*np.exp(-1j*omega*t)
    return v.real, v.imag

def qubit_to_3phase(q, t, omega=1.0):
    v_alpha, v_beta = qubit_to_alphabeta(q, t, omega=omega)
    return clarke_inv(v_alpha, v_beta)

# ==========================================
# 6) CHSH experiment + plots
# ==========================================
def estimate_E_and_collect(psi, a, b, N=2000, seed=0, **trial_kw):
    rng = np.random.default_rng(seed)
    sprod = np.empty(N, dtype=float)
    counts = {(+1,+1):0,(+1,-1):0,(-1,+1):0,(-1,-1):0}
    examples = []

    for k in range(N):
        r = run_trial(psi, a, b, seed=int(rng.integers(0, 2**31-1)), **trial_kw)
        sprod[k] = r["sA"] * r["sB"]
        counts[(r["sA"], r["sB"])] += 1
        if len(examples) < 4:
            examples.append(r)

    E = sprod.mean()
    return E, counts, examples

def run_chsh_full(N=2000, seed=1, **trial_kw):
    psi = bell_singlet()
    a0, a1 = 0.0, np.pi/4
    b0, b1 = np.pi/8, -np.pi/8

    E00, C00, ex00 = estimate_E_and_collect(psi, a0, b0, N=N, seed=seed+0, **trial_kw)
    E01, C01, ex01 = estimate_E_and_collect(psi, a0, b1, N=N, seed=seed+1, **trial_kw)
    E10, C10, ex10 = estimate_E_and_collect(psi, a1, b0, N=N, seed=seed+2, **trial_kw)
    E11, C11, ex11 = estimate_E_and_collect(psi, a1, b1, N=N, seed=seed+3, **trial_kw)

    S = abs(E00 + E01 + E10 - E11)
    return dict(
        psi=psi,
        settings=(a0,a1,b0,b1),
        E=(E00,E01,E10,E11),
        S=S,
        counts=(C00,C01,C10,C11),
        examples=(ex00,ex01,ex10,ex11),
        trial_kw=trial_kw
    )

# --------------------
# RUN IT
# --------------------
res = run_chsh_full(N=1500, seed=10, T=6.0, dt=0.003, sigma=0.7, sep=3.2, kphase=1.1)
E00,E01,E10,E11 = res["E"]
S = res["S"]
a0,a1,b0,b1 = res["settings"]
print("E00,E01,E10,E11 =", [round(x,4) for x in res["E"]])
print("CHSH S =", round(S,4), " (2√2 =", round(2*np.sqrt(2),4), ")")

# ==========================================
# Plot 1: CHSH correlations
# ==========================================
plt.figure(figsize=(8,3.8))
plt.bar(["E(a0,b0)","E(a0,b1)","E(a1,b0)","E(a1,b1)"], [E00,E01,E10,E11])
plt.axhline(0, lw=1)
plt.grid(True, alpha=0.3)
plt.title(f"Bohmian pointer model: strict ±1 outcomes\nS = {S:.3f} (classical ≤2, Tsirelson 2√2≈{2*np.sqrt(2):.3f})")
plt.tight_layout()
plt.show()

# ==========================================
# Plot 2: Branch-count histograms vs Born probs
# ==========================================
def plot_counts_vs_born(title, counts, born_probs):
    order = [(+1,+1),(+1,-1),(-1,+1),(-1,-1)]
    obs = np.array([counts[o] for o in order], dtype=float)
    obs = obs / obs.sum()
    x = np.arange(4)
    plt.figure(figsize=(7.5,3.2))
    plt.bar(x-0.15, born_probs, width=0.3, label="Born probs")
    plt.bar(x+0.15, obs, width=0.3, label="Observed (pointer outcomes)")
    plt.xticks(x, ["(+,+)","(+,-)","(-,+)","(-,-)"])
    plt.grid(True, alpha=0.3)
    plt.title(title)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

# use the "Born probs" from first example run per setting (they’re essentially fixed per (a,b))
def born_probs_for_setting(psi, a, b):
    _, _, probs = branch_coeffs_and_kets(psi, a, b)
    return probs

psi = res["psi"]
born00 = born_probs_for_setting(psi, a0, b0)
born01 = born_probs_for_setting(psi, a0, b1)
born10 = born_probs_for_setting(psi, a1, b0)
born11 = born_probs_for_setting(psi, a1, b1)

C00,C01,C10,C11 = res["counts"]
plot_counts_vs_born(f"Counts vs Born: (a0={np.degrees(a0):.1f}°, b0={np.degrees(b0):.1f}°)", C00, born00)
plot_counts_vs_born(f"Counts vs Born: (a0={np.degrees(a0):.1f}°, b1={np.degrees(b1):.1f}°)", C01, born01)
plot_counts_vs_born(f"Counts vs Born: (a1={np.degrees(a1):.1f}°, b0={np.degrees(b0):.1f}°)", C10, born10)
plot_counts_vs_born(f"Counts vs Born: (a1={np.degrees(a1):.1f}°, b1={np.degrees(b1):.1f}°)", C11, born11)

# ==========================================
# Plot 3: Example pointer trajectories + abc waveforms for the realized branch
# ==========================================
def plot_examples(examples, a_deg, b_deg, omega=1.0):
    t = np.linspace(0, 2*np.pi, 800, endpoint=False)
    for i, r in enumerate(examples[:3]):
        sA, sB = r["sA"], r["sB"]
        branches = r["branches"]
        # pick local eigenkets matching the realized (sA,sB)
        vA = None; vB = None
        for (SA,SB,ketA,ketB) in branches:
            if SA==sA and SB==sB:
                vA, vB = ketA, ketB

        vaA, vbA, vcA = qubit_to_3phase(vA, t, omega=omega)
        vaB, vbB, vcB = qubit_to_3phase(vB, t, omega=omega)

        fig, axs = plt.subplots(2, 2, figsize=(13.5,6))
        axs[0,0].plot(r["ts"], r["yA_tr"]); axs[0,0].axhline(0, ls="--", lw=1)
        axs[0,0].grid(True, alpha=0.3); axs[0,0].set_title(f"Alice pointer yA(t) → sA={sA:+d}")

        axs[0,1].plot(r["ts"], r["yB_tr"]); axs[0,1].axhline(0, ls="--", lw=1)
        axs[0,1].grid(True, alpha=0.3); axs[0,1].set_title(f"Bob pointer yB(t) → sB={sB:+d}")

        axs[1,0].plot(t, vaA, label="Va"); axs[1,0].plot(t, vbA, label="Vb"); axs[1,0].plot(t, vcA, label="Vc")
        axs[1,0].grid(True, alpha=0.3); axs[1,0].legend(frameon=False, fontsize=8)
        axs[1,0].set_title("Alice abc waveform (rendered eigenket)")

        axs[1,1].plot(t, vaB, label="Va"); axs[1,1].plot(t, vbB, label="Vb"); axs[1,1].plot(t, vcB, label="Vc")
        axs[1,1].grid(True, alpha=0.3); axs[1,1].legend(frameon=False, fontsize=8)
        axs[1,1].set_title("Bob abc waveform (rendered eigenket)")

        fig.suptitle(f"Example run {i+1}: settings (a={a_deg:.1f}°, b={b_deg:.1f}°) → branch ({sA:+d},{sB:+d})", y=1.02)
        plt.tight_layout()
        plt.show()

ex00, ex01, ex10, ex11 = res["examples"]
plot_examples(ex00, np.degrees(a0), np.degrees(b0))
plot_examples(ex01, np.degrees(a0), np.degrees(b1))
plot_examples(ex10, np.degrees(a1), np.degrees(b0))
plot_examples(ex11, np.degrees(a1), np.degrees(b1))
